# Using the Resume Feature and Iteration Loggers in QMCPy

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/resume_examples.ipynb`](../../../QMCPy/demos/demo_resume_data/resume_examples.ipynb)

This Julia notebook keeps the same compact, recipe-style structure while using the `QMC.jl` APIs for resumable integration and iteration logging.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/demo_resume_data/resume_examples.ipynb)

In [1]:
using QMC
using Serialization
using Printf


## Step 1: Quick Estimate (Loose Tolerance)

Define a 3-D Genz integral over the unit cube and run `CubQMCLatticeG` with a loose tolerance to get a fast initial estimate. We fix the seed so notebook output is reproducible.


In [2]:
function make_solver(; abs_tol=1e-3, rel_tol=0.0, seed=7, dimension=3)
    dd = Lattice(dimension; seed=seed)
    tm = Uniform(dd)
    f = Genz(tm; kind=:oscillatory, a=ones(dimension), u=0.5 .* ones(dimension))
    return CubQMCLatticeG(f; abs_tol=abs_tol, rel_tol=rel_tol, trace_iterations=true)
end

solver_loose = make_solver(abs_tol=1e-3)
result1 = integrate(solver_loose)
@printf("Loose run: solution = %.8f, n_total = %d, error_bound = %.3e
", result1.solution, result1.data[:n_total], result1.data[:error_bound])
iterations(result1.data[:iteration_log])


Loose run: solution = -0.06242316, n_total = 4096, error_bound = 6.555e-04


3-element Vector{@NamedTuple{iter::Int64, n::Int64, solution::Float64, error_bound::Float64, tol::Float64, elapsed::Float64}}:
 (iter = 1, n = 1024, solution = -0.062397714628342794, error_bound = 0.002656080832381239, tol = 0.001, elapsed = 0.017634153366088867)
 (iter = 2, n = 2048, solution = -0.062498224196483126, error_bound = 0.0013826990230159672, tol = 0.001, elapsed = 0.017712116241455078)
 (iter = 3, n = 4096, solution = -0.062423156388330146, error_bound = 0.0006554973268554603, tol = 0.001, elapsed = 0.017870187759399414)

## Step 2: Save the Integration State

In `QMC.jl`, the resume payload is the solver data dictionary returned by `integrate`. We save it to disk with Julia serialization so a later run can pick up from the same state.


In [3]:
output_dir = joinpath(pwd(), "output")
mkpath(output_dir)
save_path = joinpath(output_dir, "resume_example_data1.jls")
serialize(save_path, result1.data)
println("Saved resume data to: ", save_path)


Saved resume data to: /Users/terrya/Documents/ProgramData/QMCSoftware_space/QMC.jl/demos/demo_resume_data/output/resume_example_data1.jls

## Step 3: Resume with a Tighter Tolerance

Load the saved state and resume with a compatible solver. Only the extra work needed for the tighter tolerance is generated.


In [4]:
loaded_data = deserialize(save_path)
solver_tight = make_solver(abs_tol=2.5e-4)
result2 = integrate(solver_tight; resume=loaded_data)
extra_samples = result2.data[:n_total] - result1.data[:n_total]
@printf("Resumed run: solution = %.8f, n_total = %d, added samples = %d
", result2.solution, result2.data[:n_total], extra_samples)
iterations(result2.data[:iteration_log])


Resumed run: solution = -0.06230787, n_total = 16384, added samples = 12288


2-element Vector{@NamedTuple{iter::Int64, n::Int64, solution::Float64, error_bound::Float64, tol::Float64, elapsed::Float64}}:
 (iter = 1, n = 8192, solution = -0.062328353432913125, error_bound = 0.0003798348489505347, tol = 0.00025, elapsed = 0.018862009048461914)
 (iter = 2, n = 16384, solution = -0.062307874564558795, error_bound = 0.00020323683885610264, tol = 0.00025, elapsed = 0.03201484680175781)

## Step 4: Compare with a Fresh Run

For reference, solve the same tighter problem from scratch and compare the result with the resumed workflow.


In [5]:
fresh_tight = integrate(make_solver(abs_tol=2.5e-4))
@printf("Fresh tight run: solution = %.8f, n_total = %d
", fresh_tight.solution, fresh_tight.data[:n_total])
@printf("Absolute difference between resumed and fresh solutions: %.3e
", abs(result2.solution - fresh_tight.solution))

@assert abs(result2.solution - fresh_tight.solution) ≤ result2.data[:error_bound] + fresh_tight.data[:error_bound]
@assert result2.data[:n_total] == fresh_tight.data[:n_total]
@assert !isempty(iterations(result2.data[:iteration_log]))


Fresh tight run: solution = -0.06230787, n_total = 16384
Absolute difference between resumed and fresh solutions: 0.000e+00


## Step 5: Review the Stored Iteration Log

The iteration log is available even if you do not print it live. This makes it easy to inspect the stopping history after the solve completes.


In [6]:
for row in iterations(result2.data[:iteration_log])
    println(row)
end


(iter

 = 1, n = 8192, solution = -0.062328353432913125, error_bound = 0.0003798348489505347, tol = 0.00025, elapsed = 0.018862009048461914)
(iter = 2, n = 16384, solution = -0.062307874564558795, error_bound = 0.00020323683885610264, tol = 0.00025, elapsed = 0.03201484680175781)
